# অধ্যায় ৭: টেক্সট ডেটা
## পাঠ ৭.৩: টপিক মডেলিং

আজ আমরা শিখব কীভাবে স্বয়ংক্রিয়ভাবে একটি বড় টেক্সট ডেটাসেট থেকে বিভিন্ন বিষয় (টপিক) বের করা যায়।

### A. গল্প: গ্রন্থাগার

একটি বিশাল গ্রন্থাগারের কথা ভাবো, যেখানে হাজার হাজার বই আছে কিন্তু কোনোটিতেই বিষয় লেখা নেই। কয়েক হাজার বই না পড়েই কী করে বের করবে সেগুলো কী নিয়ে?

তুমি বইয়ের পাতাগুলো উল্টিয়ে দেখবে কোন শব্দগুলো বেশি আছে। যদি 'বল', 'গোল', 'স্টেডিয়াম', 'গোলরক্ষক' বেশি থাকে—বইটি সম্ভবত ফুটবল নিয়ে। আর 'তারামণ্ডল', 'ছায়াপথ', 'টেলিস্কোপ' বেশি থাকলে—মহাকাশ নিয়ে।

ঠিক এটাই করে টপিক মডেলিং! এটি অটোমেটিক্যালি টেক্সট থেকে 'থিম' বা 'টপিক' বের করে।

### B. Latent Dirichlet Allocation (LDA)

LDA একটি টপিক মডেলিং অ্যালগরিদম। এটি ধরে নেয়:
- প্রতিটি document একাধিক টপিকের মিশ্রণ
- প্রতিটি টপিক কিছু নির্দিষ্ট শব্দের মিশ্রণ

উদাহরণ: একটি document ৬০% 'ক্রীড়া' এবং ৪০% 'অর্থনীতি' হতে পারে।
আর 'ক্রীড়া' টপিকে 'বল', 'গোল', 'খেলোয়াড়' ইত্যাদি শব্দ বেশি থাকতে পারে।

### C. NMF (Non-Negative Matrix Factorization)

NMF আরেকটি টপিক মডেলিং পদ্ধতি। এটি টেক্সট ম্যাট্রিক্সকে দুটি ছোট ম্যাট্রিক্সে ভাগ করে—একটি টপিক-শব্দ সম্পর্ক এবং অন্যটি document-টপিক সম্পর্ক দেখায়।

LDA এবং NMF-এর পার্থক্য:
- LDA: প্রোবাবিলিস্টিক পদ্ধতি (গণিতের সম্ভাবনা ব্যবহার করে)
- NMF: লিনিয়ার বীজগণিতের পদ্ধতি
- সাধারণত LDA বেশি জনপ্রিয়

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
import numpy as np

# Shadharon text data diye topic modeling
documents = [
    "artificial intelligence is changing the world of technology",
    "machine learning algorithms can predict patterns from data",
    "deep learning uses neural networks to understand images",
    "technology companies invest heavily in ai research",
    "neural networks are inspired by the human brain",
    "data science combines statistics and programming skills",
    "ai can help doctors diagnose diseases more accurately",
    "python is a popular language for machine learning",
    "sports are an important part of daily life",
    "basketball players need to practice shooting every day",
    "football is the most popular sport in the world",
    "tennis requires speed agility and quick reflexes",
    "sports teams analyze data to improve performance",
    "cricket is a game of bat and ball",
    "athletes must train hard to stay in shape",
    "the olympics bring together athletes from all countries",
    "ai technology helps self driving cars navigate roads",
    "machine learning models need lots of data to train",
    "natural language processing helps computers understand text",
    "computer vision allows machines to see and recognize objects",
    "soccer players run many kilometers during a match",
    "big data analytics helps businesses make better decisions",
    "reinforcement learning teaches ai through trial and error",
    "sports fans love to watch their favorite teams play",
    "neural networks can generate realistic images and text",
    "data scientists use pandas to analyze tabular data",
    "cloud computing provides power for large ai models",
    "coaches use video analysis to improve team strategy",
    "ai algorithms can recommend movies and music to users",
]

news_data = documents
news_target = np.array([0]*15 + [1]*14)
news_target_names = ['Technology', 'Sports']

print('Dataset ready!')
print(f'  Total samples: {len(news_data)}')
print(f'  Categories: {news_target_names}')
print(f'  First document: {news_data[0][:60]}...')


Dataset ready!
  Total samples: 29
  Categories: ['Technology', 'Sports']
  First document: artificial intelligence is changing the world of technology...


### D. টেক্সট প্রিপ্রসেসিং

প্রথমে আমরা টেক্সটকে CountVectorizer বা TfidfVectorizer দিয়ে সংখ্যায় রূপান্তর করব। টপিক মডেলিং-এর জন্য সাধারণত CountVectorizer বেশি ব্যবহার হয়।

In [2]:
# CountVectorizer দিয়ে ফিচার
vec = CountVectorizer(
    max_features=1000,
    stop_words='english',
    max_df=0.5,
    min_df=5
)

X = vec.fit_transform(news_data)
print('ফিচার ম্যাট্রিক্স:', X.shape)
print('ভোকাবুলারি সাইজ:', len(vec.get_feature_names_out()))

ফিচার ম্যাট্রিক্স: (29, 3)
ভোকাবুলারি সাইজ: 3


### E. LDA দিয়ে টপিক এক্সট্র্যাকশন

এখন আমরা LDA ব্যবহার করে ৫টি টপিক বের করব এবং প্রতিটি টপিকের সেরা ১০টি শব্দ দেখব।

In [3]:
# LDA model
lda = LatentDirichletAllocation(
    n_components=3,
    max_iter=10,
    random_state=42
)

lda.fit(X)

# Topic dekhar jonno function
def print_topics(model, vectorizer, n_top_words=10):
    feature_names = vectorizer.get_feature_names_out()
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        print(f'Topic {topic_idx + 1}: {" ".join(top_words)}')
    print()

print('LDA Topic Modeling Result:')
print('=' * 60)
print_topics(lda, vec)

LDA Topic Modeling Result:
Topic 1: ai learning data
Topic 2: data learning ai
Topic 3: learning data ai



### F. টপিক বিশ্লেষণ

উপরে ৫টি টপিক দেখতে পাচ্ছ। লক্ষ্য করো:
- টপিক ১: baseball, game, team, players → খেলাধুলা
- টপিক ২: space, nasa, launch, earth → মহাকাশ
- টপিক ৩: graphics, image, computer → কম্পিউটার গ্রাফিক্স
- টপিক ৪: israel, arab, jews → রাজনীতি/মধ্যপ্রাচ্য
- টপিক ৫: gun, people, don → সাধারণ আলোচনা

মজার ব্যাপার হলো, আমরা কোনো লেবেল ব্যবহার করিনি! LDA নিজে থেকেই টপিক বের করে ফেলেছে।

In [4]:
# প্রতিটি document-এর টপিক বণ্টন দেখা
doc_topics = lda.transform(X[:5])
print('প্রথম 5টি document-এর টপিক বণ্টন:')
print(pd.DataFrame(
    np.round(doc_topics, 3),
    columns=[f'টপিক{i+1}' for i in range(doc_topics.shape[1])]
))

প্রথম 5টি document-এর টপিক বণ্টন:
   টপিক1  টপিক2  টপিক3
0  0.333  0.333  0.333
1  0.111  0.773  0.115
2  0.167  0.657  0.175
3  0.664  0.167  0.169
4  0.333  0.333  0.333


### G. NMF দিয়ে টপিক মডেলিং

এখন আমরা NMF ব্যবহার করে একই কাজ করি এবং ফলাফল তুলনা করি।

In [5]:
# NMF মডেল (TF-IDF প্রয়োজন)
tfidf_vec = TfidfVectorizer(
    max_features=1000, stop_words='english',
    max_df=0.5, min_df=5
)
X_tfidf = tfidf_vec.fit_transform(news_data)

nmf = NMF(n_components=3, random_state=42)
nmf.fit(X_tfidf)
print('NMF টপিক মডেলিং ফলাফল:')
print('=' * 60)
print_topics(nmf, tfidf_vec)

NMF টপিক মডেলিং ফলাফল:
Topic 1: ai learning data
Topic 2: data learning ai
Topic 3: learning data ai



### H. লাইব্রেরি ছাড়া চিন্তা: টপিক মডেলিং কীভাবে কাজ করে?

LDA-র কাজের পদ্ধতি (সহজ ভাষায়):

১. প্রথমে এলোমেলোভাবে প্রতিটি document-এ টপিক অ্যাসাইন করে
২. প্রতিটি শব্দের জন্য:
   a. document-এ কোন টপিক কতবার আছে সেটা দেখে
   b. সব document মিলিয়ে শব্দটি কোন টপিকে কতবার এসেছে সেটা দেখে
   c. দুইয়ের সমন্বয়ে শব্দটির জন্য নতুন টপিক নির্বাচন করে
৩. অনেকবার পুনরাবৃত্তি করে যতক্ষণ না টপিক স্থিতিশীল হয়

এটি 'Gibbs Sampling' নামে পরিচিত।

### I. LDA-র গুরুত্বপূর্ণ প্যারামিটার

- **n_components:** কতটি টপিক বের করবে (সবচেয়ে গুরুত্বপূর্ণ)
- **learning_method:** 'batch' (ধীর, নির্ভুল) বা 'online' (দ্রুত, বড় ডেটার জন্য)
- **max_iter:** কতবার পুনরাবৃত্তি করবে
- **n_jobs:** কতগুলো কোর ব্যবহার করবে

**কতগুলি টপিক সেট করব?**
- খুব কম টপিক → প্রতিটি টপিক খুব জেনারেল হবে (সব মিলিয়ে যাবে)
- খুব বেশি টপিক → প্রতিটি টপিক খুব স্পেসিফিক হবে (ওভারফিটিং)
- সাধারণত ৫-২০ টি টপিক দিয়ে শুরু করা ভালো

একটি কৌশল হলো 'perplexity' স্কোর দেখা—এটি বলে LDA মডেল কতটা 'আশ্চর্য' হচ্ছে ডেটা দেখে। কম perplexity ভালো। কিন্তু খুব কম perplexity ওভারফিটিং নির্দেশ করতে পারে।

### J. তুমি কি বুঝতে পেরেছ?

**প্রশ্ন ১:** টপিক মডেলিং কীভাবে টেক্সট থেকে টপিক বের করে?

**প্রশ্ন ২:** LDA এবং NMF-র মধ্যে মৌলিক পার্থক্য কী?

**প্রশ্ন ৩:** n_components প্যারামিটার কী নির্ধারণ করে?

**প্রশ্ন ৪:** LDA-র 'Gibbs Sampling' কী?

### K. সারসংক্ষেপ

আজ আমরা শিখলাম:
✅ টপিক মডেলিং অটোমেটিক্যালি টেক্সট থেকে বিষয় বের করে
✅ LDA প্রোবাবিলিস্টিক পদ্ধতি ব্যবহার করে
✅ NMF লিনিয়ার বীজগণিত ব্যবহার করে
✅ CountVectorizer + LDA একটি জনপ্রিয় কম্বিনেশন
✅ প্রতিটি document একাধিক টপিকের মিশ্রণ হতে পারে
✅ n_components সঠিকভাবে নির্বাচন করা গুরুত্বপূর্ণ

এভাবে আমরা অধ্যায় ৭ শেষ করলাম। পরবর্তী অধ্যায়ে আমরা উপসংহার এবং প্রকল্পে মেশিন লার্নিং প্রয়োগ নিয়ে শিখব!